In [32]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import brier_score
from scipy.interpolate import interp1d
from sklearn.inspection import permutation_importance
from sksurv.metrics import concordance_index_censored 

import sys
import os

# Point directly to the project root (one level up from notebooks/)
sys.path.append(os.path.abspath(".."))  # '..' from notebooks/ goes to root

from src.features import make_features

In [33]:
train_df = pd.read_csv("../data/train_processed.csv")
test_df = pd.read_csv("../data/test_processed.csv")
test_event_id = test_df["event_id"].copy()

In [34]:
# Prepare data for modeling ===#

X = make_features(train_df)
X_test = make_features(test_df)

y = np.array(
    list(zip(train_df["event"].astype(bool), train_df["time_to_hit_hours"].astype(float))),
    dtype=[("event", bool), ("time", float)]
)


# Split the data into training and validation sets
X_train, X_val, y_train_event, y_val_event, y_train_time, y_val_time = train_test_split(
    X, train_df['event'], train_df['time_to_hit_hours'], test_size=0.2, random_state=42
)

In [35]:
# Create structured array for survival data
y_train = np.array(list(zip(y_train_event, y_train_time)),
                   dtype=[('event', bool), ('time', float)])
y_val = np.array(list(zip(y_val_event, y_val_time)),
                 dtype=[('event', bool), ('time', float)])

rsf = RandomSurvivalForest(
    n_estimators=2000,
    max_depth=None,
    min_samples_split=20,
    min_samples_leaf=3,
    max_features="log2",
    n_jobs=-1,
    random_state=42
)

rsf.fit(X_train, y_train)

RandomSurvivalForest(max_features='log2', min_samples_split=20,
                     n_estimators=2000, n_jobs=-1, random_state=42)

In [36]:
# 1. Gradient Boosting Survival stacking
from sksurv.ensemble import GradientBoostingSurvivalAnalysis

# Fit GBSA on same features
gbsa = GradientBoostingSurvivalAnalysis(
    n_estimators=700,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
# y_train is structured array
gbsa.fit(X_train, y_train)

# 2. Get survival curves from GBSA
surv_probs_gbsa = gbsa.predict_survival_function(X_test, return_array=True)

# 3. Combine RSF + GBSA 
surv_probs_rsf = rsf.predict_survival_function(X_test, return_array=True)

# Ensemble by averaging survival probabilities
ensemble_surv = (surv_probs_rsf + surv_probs_gbsa) / 2

# Overwrite surv_probs_test for downstream horizon extraction 
surv_probs_test = ensemble_surv

In [37]:
from sksurv.linear_model import CoxPHSurvivalAnalysis
y_train_df = pd.DataFrame({
    "event": y_train['event'],
    "time": y_train['time']
})
coxph = CoxPHSurvivalAnalysis()
coxph.fit(X_train, y_train_df.to_records(index=False))

# 2. Predict survival function for test set
unique_times = rsf.unique_times_

baseline_surv = coxph.baseline_survival_  
risk_scores = coxph.predict(X_test)        

cox_surv_test = np.zeros((X_test.shape[0], len(unique_times)))
for i, t in enumerate(unique_times):
    cox_surv_test[:, i] = baseline_surv(t) ** np.exp(risk_scores)

/home/jaren/miniconda3/envs/team2b/lib/python3.9/site-packages/sksurv/linear_model/coxph.py:449: LinAlgWarning: Ill-conditioned matrix (rcond=3.09916e-31): result may not be accurate.
  delta = solve(
/home/jaren/miniconda3/envs/team2b/lib/python3.9/site-packages/sksurv/linear_model/coxph.py:449: LinAlgWarning: Ill-conditioned matrix (rcond=3.53087e-30): result may not be accurate.
  delta = solve(
/home/jaren/miniconda3/envs/team2b/lib/python3.9/site-packages/sksurv/linear_model/coxph.py:449: LinAlgWarning: Ill-conditioned matrix (rcond=3.4385e-30): result may not be accurate.
  delta = solve(
/home/jaren/miniconda3/envs/team2b/lib/python3.9/site-packages/sksurv/linear_model/coxph.py:449: LinAlgWarning: Ill-conditioned matrix (rcond=6.43307e-30): result may not be accurate.
  delta = solve(
/home/jaren/miniconda3/envs/team2b/lib/python3.9/site-packages/sksurv/linear_model/coxph.py:449: LinAlgWarning: Ill-conditioned matrix (rcond=3.41112e-29): result may not be accurate.
  delta = sol

In [38]:
#Permutation importance
result = permutation_importance(
    rsf,
    X_train,
    y_train,
    n_repeats=10,
    random_state=42,
)

importances = result.importances_mean

for f, imp in sorted(zip(X_train.columns, importances),
                     key=lambda x: x[1],
                     reverse=True)[:10]:
    print(f"{f}: {imp:.4f}")

dist_min_ci_0_5h: 0.0180
inv_dist_min: 0.0142
time_to_collision_est: 0.0043
threat_projection: 0.0024
num_perimeters_0_5h: 0.0020
directed_growth: 0.0015
dt_first_last_0_5h: 0.0013
alignment_abs: 0.0010
closing_speed_m_per_h: 0.0000
spread_bearing_cos: -0.0002


In [39]:
# Horizons for test predictions
times = np.array([12, 24, 48, 72])
time_indices = []
for t in times:
    idx = np.searchsorted(rsf.unique_times_, t, side="right") - 1
    idx = np.clip(idx, 0, len(rsf.unique_times_) - 1)
    time_indices.append(idx)

# Ensemble predictions (RSF + GBSA) 
surv_rsf_test = rsf.predict_survival_function(X_test, return_array=True)
surv_gbsa_test = gbsa.predict_survival_function(X_test, return_array=True)

w_rsf = 0.7
w_gbsa = 0.15 
w_cox = 0.05
#surv_test_ensemble = w_rsf * surv_rsf_test + w_gbsa * surv_gbsa_test
surv_test_ensemble = (
    w_rsf * surv_rsf_test + w_gbsa * surv_gbsa_test + w_cox * cox_surv_test
)
# Extract probabilities at each horizon 
probs_test = {t: 1 - surv_test_ensemble[:, idx] for t, idx in zip(times, time_indices)}

def clip_probs(p):
    return np.clip(p, 1e-4, 1 - 1e-4)

probs_12h = clip_probs(probs_test[12.0])
probs_24h = clip_probs(probs_test[24.0])
probs_48h = clip_probs(probs_test[48.0])
probs_72h = clip_probs(probs_test[72.0])

# Verify lengths
print(len(probs_12h), len(test_df))

95 95


In [40]:
sub = pd.DataFrame({
    "event_id": test_event_id.astype(np.int64),
    "prob_12h": probs_12h,
    "prob_24h": probs_24h,
    "prob_48h": probs_48h,
    "prob_72h": probs_72h,
})

In [41]:
sub.to_csv("../results/submission_final.csv", index=False)
print("Saved: submission_final.csv")
print(sub.head())

Saved: submission_final.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.101019  0.103201  0.104574  0.116978
1  13353600  0.548928  0.875838  0.936222  0.992785
2  13942327  0.101029  0.103254  0.104657  0.117294
3  16112781  0.641448  0.887675  0.940039  0.994291
4  17132808  0.261785  0.275444  0.279654  0.296381
